## Toy Beacon Localization Problem

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
def get_noisy_measurement(
    x_true, xy_beacons, meas_noise, with_noise=True, delay=0.0, c=1.0
):
    num_beacons = xy_beacons.shape[0]
    ranges = np.zeros(num_beacons)
    for i in range(num_beacons):
        ranges[i] = (
            np.linalg.norm(xy_beacons[i, :] - x_true[0:2]) + c * x_true[2]
        )  # clock bias
        if i <= 1:
            ranges[i] += delay  # no delay for first two beacons
    if with_noise:
        ranges += np.random.normal(0, meas_noise, num_beacons)
    return ranges


def get_est_measurement(x_est, xy_beacons, meas_noise, c=1.0):
    num_beacons = xy_beacons.shape[0]
    z_est = np.zeros(num_beacons)
    H = np.zeros((num_beacons, 3))  # states are [x, y, clock_bias]
    R = np.eye(num_beacons) * meas_noise**2  # measurement noise covariance
    for i in range(num_beacons):
        d = np.linalg.norm(xy_beacons[i, :] - x_est[:2])
        z_est[i] = d + c * x_est[2]  # clock bias
        H[i, 0] = (x_est[0] - xy_beacons[i, 0]) / d
        H[i, 1] = (x_est[1] - xy_beacons[i, 1]) / d
        H[i, 2] = c  # clock bias term
    return z_est, H, R


def measurement_update(x_est, P, z_true, z_est, H, R):
    y = z_true - z_est  # measurement residual
    S = H @ P @ H.T + R  # residual covariance
    K = P @ H.T @ np.linalg.inv(S)  # Kalman gain
    x_upd = x_est + K @ y
    J = np.eye(len(x_est)) - K @ H
    P_upd = J @ P  # simplified covariance update
    P_upd = J @ P @ J.T + K @ R @ K.T  # Joseph form
    return x_upd, P_upd, S


def plot_error_ellipse(x_est, P, ax, n_std=2.0, **kwargs):
    from matplotlib.patches import Ellipse

    cov_xy = P[0:2, 0:2]
    eigvals, eigvecs = np.linalg.eig(cov_xy)
    order = eigvals.argsort()[::-1]
    eigvals = eigvals[order]
    eigvecs = eigvecs[:, order]
    angle = np.arctan2(eigvecs[1, 0], eigvecs[0, 0]) * 180 / np.pi
    width, height = 2 * n_std * np.sqrt(eigvals)
    ellipse = Ellipse(
        xy=(x_est[0], x_est[1]), width=width, height=height, angle=angle, **kwargs
    )
    ax.add_patch(ellipse)

    return ax


def plot_maps(fig, ax, xy_beacons, x_true):
    ax.scatter(
        xy_beacons[:, 0], xy_beacons[:, 1], marker="^", color="red", label="Beacons"
    )
    ax.plot(x_true[0], x_true[1], "go", label="True Position")
    ax.set_xlabel("X Position")
    ax.set_ylabel("Y Position")
    ax.set_title("Position Estimation")
    ax.legend()
    ax.axis("equal")
    return fig, ax

In [ ]:
def run_simulation(
    x_est0,
    x_true,
    P0,
    xy_beacons,
    num_iterations=1,
    meas_noise=5.0,
    delay=0.0,
    c=1.0,
    seed=0,
):
    nx = len(x_est0)
    ny = xy_beacons.shape[0]
    np.random.seed(seed)

    x_est = x_est0.copy()
    P = P0.copy()
    x_est_history = np.zeros((num_iterations, nx))
    P_history = np.zeros((num_iterations, P.shape[0], P.shape[1]))
    x_est_err = np.zeros((num_iterations, nx))
    S_history = np.zeros((num_iterations, ny, ny))

    print("Starting Measurement Update Iterations")
    print(
        "Iteration 0: Estimation Error: {:.2f}".format(np.linalg.norm(x_est - x_true))
    )
    for i in range(num_iterations):
        z_est, H, R = get_est_measurement(x_true, xy_beacons, meas_noise, c=c)
        z_true = get_noisy_measurement(
            x_true, xy_beacons, meas_noise, with_noise=True, delay=delay, c=c
        )
        x_est, P, S = measurement_update(x_est, P, z_true, z_est, H, R)
        x_est_history[i] = x_est.copy()
        P_history[i] = P.copy()
        S_history[i] = S.copy()
        x_est_err[i] = x_est - x_true
        print(
            f"Iteration {i+1}: Estimation Error: {np.linalg.norm(x_est_err[i, 0:2])}, condition number of P: {np.linalg.cond(P):.5e}, condition number of S: {np.linalg.cond(S):.5e}"
        )
        # print("P_est:\n", P)
        # print("S:\n", S)
        # print("H:\n", H)

    fig, ax = plt.subplots(figsize=(6, 6))
    fig, ax = plot_maps(fig, ax, xy_beacons, x_true)
    ax = plot_error_ellipse(
        x_est0,
        P0,
        ax,
        edgecolor="red",
        facecolor="red",
        alpha=0.3,
        label="Initial Uncertainty",
    )
    ax.plot(x_est0[0], x_est0[1], "rx", label="Initial Estimate")

    ax = plot_error_ellipse(
        x_est_history[-1],
        P_history[-1],
        ax,
        edgecolor="blue",
        facecolor="blue",
        alpha=0.3,
        label="Updated Uncertainty",
    )
    ax.plot(x_est_history[-1, 0], x_est_history[-1, 1], "bx", label="Estimation")
    ax.grid(True)
    ax.legend()
    plt.show()

## Good Geometry Example

In [ ]:
# place 3 beacons
xy_beacons = np.array([[0.0, 0.0], [100.0, 0.0], [0.0, 100.0], [100.0, 100.0]])
# true state [x, y, clock_bias]
x_true = np.array([50.0, 50.0, 10.0])
P0 = np.diag([100.0, 100.0, 100.0])  # initial covariance

# initial estimate
# sample from P0 around x_true
x_est0 = np.random.multivariate_normal(x_true, P0)
# x_est0 = np.array([50.0, 50.0, 0.0])

run_simulation(
    x_est0,
    x_true,
    P0,
    xy_beacons,
    num_iterations=3,
    meas_noise=10,
    delay=0.0,
    c=1.0,
    seed=0,
)

## Poor Geometry Case

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
fig, ax = plot_maps(fig, ax, xy_beacons, x_true)

In [ ]:
# place 3 beacons
xy_beacons = np.zeros((4, 2))
sigma_xy = 1.0
for i in range(4):
    xy_beacons[i, 0] = 100 + np.random.normal(0, sigma_xy)
    xy_beacons[i, 1] = 100 + np.random.normal(0, sigma_xy)

run_simulation(
    x_est0,
    x_true,
    P0,
    xy_beacons,
    num_iterations=2,
    meas_noise=1e-3,
    delay=0.0,
    c=1.0,
    seed=0,
)

## Poor Geometry with Bias

In [ ]:
# place 3 beacons
np.random.seed(0)
run_simulation(
    x_est0,
    x_true,
    P0,
    xy_beacons,
    num_iterations=2,
    meas_noise=1e-3,
    delay=1e-1,
    c=1.0,
    seed=0,
)